## Step 1. Check and display the structure of the main dictionary with all postcodes 

In [0]:
code_df = spark.read.format("parquet").option("header", "true").option("inferSchema", "true").load("dbfs:/mnt/bronze/codes_geo.parquet")
display(code_df.describe())

summary,straat,huisnummer,huisletter,huistoevoeging,woonplaats,postcode,x,y,lon,lat,oppervlakte,gebruiksdoelen,bouwjaar,id
count,9657951,9657951,9657951,9657951,9657951,9657951,9657951,9657951,9657951,9657951,9657951,9657951,9657951,9657951
mean,null,99.32074929765123,null,3094.9349047473333,null,null,147292.7534647861,456430.1703641286,5.277091201875913,52.09354101335212,178.58427072160544,null,1972.8722680411197,4828976.0
stddev,null,704.2034557923242,null,1425058.495259355,null,null,52613.19313960755,58005.894931036506,0.7695214648642487,0.5209566155406219,2654.7171333738465,null,42.1480191530564,2788010.4491727566
min,'n Mös,1,,,'s Gravenmoer,,-1.0,-1.0,3.3135717,47.97476,1,"[""bijeenkomstfunctie"",""celfunctie"",""gezondheidszorgfunctie"",""woonfunctie""]",1000,1
max,Örehof,99999,z,zw1,de Woude,9999XL,277670.75,612642.0,7.2222476,53.49727,999999,"[""woonfunctie""]",2028,9657951


In [0]:
code_df.show(5)

+----------+----------+----------+--------------+----------+--------+---------+---------+---------+---------+-----------+--------------------+--------+-------+
|    straat|huisnummer|huisletter|huistoevoeging|woonplaats|postcode|        x|        y|      lon|      lat|oppervlakte|      gebruiksdoelen|bouwjaar|     id|
+----------+----------+----------+--------------+----------+--------+---------+---------+---------+---------+-----------+--------------------+--------+-------+
|  Ringdijk|         6|         a|              |     Graft|  1484PC|115584.34|508166.06| 4.805959|52.559685|        140|     ["woonfunctie"]|    1998|9515946|
|  Ringdijk|         7|          |              |     Graft|  1484PC|115614.28|507651.25|4.8064613| 52.55506|         18|["overige gebruik...|    1987|9515947|
|  Ringdijk|         8|          |              |     Graft|  1484PC|115654.76| 507635.3|4.8070602| 52.55492|        232|     ["woonfunctie"]|    1902|9515948|
|Noordeinde|         1|          |      

## Step 2. Clean data. Leave only "woonfunctie" facility type in column "gebruiksdoelen"

In [0]:
display(code_df.select("gebruiksdoelen").distinct().show(10))

display(code_df.filter(code_df.gebruiksdoelen.contains("woonfunctie")).show(10))

+--------------------+
|      gebruiksdoelen|
+--------------------+
|  ["kantoorfunctie"]|
|   ["winkelfunctie"]|
|["overige gebruik...|
|["kantoorfunctie"...|
|["winkelfunctie",...|
|["logiesfunctie",...|
|["overige gebruik...|
|["industriefuncti...|
|["bijeenkomstfunc...|
|["bijeenkomstfunc...|
+--------------------+
only showing top 10 rows
+----------+----------+----------+--------------+----------+--------+----------+---------+---------+---------+-----------+--------------------+--------+-------+
|    straat|huisnummer|huisletter|huistoevoeging|woonplaats|postcode|         x|        y|      lon|      lat|oppervlakte|      gebruiksdoelen|bouwjaar|     id|
+----------+----------+----------+--------------+----------+--------+----------+---------+---------+---------+-----------+--------------------+--------+-------+
|  Ringdijk|         6|         a|              |     Graft|  1484PC| 115584.34|508166.06| 4.805959|52.559685|        140|     ["woonfunctie"]|    1998|9515946|
|  Ringdi

## Step 3 Add new column 'woning'
Manual deviation by living space to identify the potential target group 

  **Vrijstaande  >=200 "oppervlakte"  - TARGET GROUP**

    Rijwoning  >=100 and <200 "oppervlakte"
    Appartement <100 "oppervlakte"

In [0]:
from pyspark.sql.functions import when

if "woning" in code_df.columns:
    code_df = code_df.drop("woning")
    
code_df = code_df.withColumn(
    "woning",
    when(
        (code_df["oppervlakte"] >= 100) & (code_df["oppervlakte"] < 200),
        "Rijwoning (100-200m2)"
    ).when(
        code_df["oppervlakte"] >= 200,
        "Vrijstaande (>200m2)"
    ).otherwise("Appartement (<100m2)")
)
display(code_df.show(10))

+----------+----------+----------+--------------+----------+--------+----------+---------+---------+---------+-----------+--------------------+--------+-------+--------------------+
|    straat|huisnummer|huisletter|huistoevoeging|woonplaats|postcode|         x|        y|      lon|      lat|oppervlakte|      gebruiksdoelen|bouwjaar|     id|              woning|
+----------+----------+----------+--------------+----------+--------+----------+---------+---------+---------+-----------+--------------------+--------+-------+--------------------+
|  Ringdijk|         6|         a|              |     Graft|  1484PC| 115584.34|508166.06| 4.805959|52.559685|        140|     ["woonfunctie"]|    1998|9515946|Rijwoning (100-20...|
|  Ringdijk|         7|          |              |     Graft|  1484PC| 115614.28|507651.25|4.8064613| 52.55506|         18|["overige gebruik...|    1987|9515947|Appartement (<100m2)|
|  Ringdijk|         8|          |              |     Graft|  1484PC| 115654.76| 507635.3|

In [0]:
from pyspark.sql.functions import sum, count, from_json, array_contains, col, size, StringType, ArrayType

# allen woonfunctie
code_df = code_df.withColumn("gebruiksdoelen_array", from_json(code_df.gebruiksdoelen, ArrayType(StringType()))).filter((array_contains(col("gebruiksdoelen_array"), "woonfunctie")) & (size(col("gebruiksdoelen_array")) == 1))

display(
    code_df.groupBy("woning").agg(
        count("id").alias("cnt_records")
    ),
    code_df.agg(count("id")).alias("total")
)

woning,cnt_records
Rijwoning (100-200m2),4008217
Appartement (<100m2),3762685
Vrijstaande (>200m2),608769


Databricks visualization. Run in Databricks to view.

# SQL PATH

In [0]:
display(spark.sql("SHOW CATALOGS"))
display(spark.sql("SHOW SCHEMAS"))
# display(spark.sql("SHOW TABLES IN databricks_dussbmw_prod1.information_schema"))

catalog
databricks_dussbmw_prod1
hive_metastore
samples
system


databaseName
default
information_schema


In [0]:

# Number of unique records (address)
code_df.createOrReplaceTempView("codes_geo_temp")


df = spark.sql("""
    SELECT 
        count(*)
    FROM codes_geo_temp
""")
display(df)

count(1)
9657951


In [0]:
# Check the location  lon and lat 
display(code_df
    .filter((code_df["huisnummer"] == 150) & (code_df["postcode"] == "7332AW"))
    .select("lon")
    .first())

display(code_df
    .filter((code_df["huisnummer"] == 150) & (code_df["postcode"] == "7332AW"))
    .select("lat")
    .first())

Row(lon=5.981220722198486)

Row(lat=52.182167053222656)

#Add google map location

In [0]:
duss_bmw_office_postcode_df = code_df.filter((code_df["huisnummer"] == 150) & (code_df["postcode"] == "7332AW"))
duss_bmw_office_postcode_df.display()

my_house_postcode_df = code_df.filter(
    (code_df["huisnummer"] == 116) & (code_df["postcode"] == "7556LC")
)
display(my_house_postcode_df)

straat,huisnummer,huisletter,huistoevoeging,woonplaats,postcode,x,y,lon,lat,oppervlakte,gebruiksdoelen,bouwjaar,id,woning
Kayersdijk,150,,,Apeldoorn,7332AW,195626.16,466169.66,5.9812207,52.182167,1064,"[""industriefunctie""]",2005,6643788,Vrijstaande


straat,huisnummer,huisletter,huistoevoeging,woonplaats,postcode,x,y,lon,lat,oppervlakte,gebruiksdoelen,bouwjaar,id,woning
Jan Tooropstraat,116,,,Hengelo,7556LC,250816.22,477768.22,6.791304,52.279564,120,"[""woonfunctie""]",2022,6915450,Rijwoning


In [0]:
from pyspark.sql.functions import radians, sin, cos, atan2, sqrt, lit

# Extract scalar values for lat2 and lon2
lat2_row = (
    code_df
    .filter((code_df["huisnummer"] == 150) & (code_df["postcode"] == "7332AW"))
    .select("lat")
    .first()
)
lon2_row = (
    code_df
    .filter((code_df["huisnummer"] == 150) & (code_df["postcode"] == "7332AW"))
    .select("lon")
    .first()
)

lat2 = lat2_row["lat"] if lat2_row is not None else None
lon2 = lon2_row["lon"] if lon2_row is not None else None

# Radius of the Earth in kilometers
R = 6371.0


my_house_postcode_df = my_house_postcode_df.withColumn(
    "distance_km",
    R * 2 * atan2(
        sqrt(
            sin((radians(lit(lat2)) - radians(my_house_postcode_df["lat"])) / 2) ** 2 +
            cos(radians(my_house_postcode_df["lat"])) * cos(radians(lit(lat2))) *
            sin((radians(lit(lon2)) - radians(my_house_postcode_df["lon"])) / 2) ** 2
        ),
        sqrt(
            1 - (
                sin((radians(lit(lat2)) - radians(my_house_postcode_df["lat"])) / 2) ** 2 +
                cos(radians(my_house_postcode_df["lat"])) * cos(radians(lit(lat2))) *
                sin((radians(lit(lon2)) - radians(my_house_postcode_df["lon"])) / 2) ** 2
            )
        )
    )
)
display(my_house_postcode_df.select("distance_km")
    .first())

Row(distance_km=56.22318302967994)

##The calculated distance is 56.2 km, which is very close and confirms the Google Maps result 56.4 km.
## This approach works to determine the distance and number of “Vrijstaande woning” from each dealership center.

![Google map](../img/distance.png)

![](path)